# Kaggle bulk uploader — iPhone + Google Drive

1. Put `Kaggle_Notebooks_FLAT_25.zip` in **My Drive**.
2. Open this notebook in Google Colab using Chrome on your iPhone. Choose **Runtime → Run all**.
3. Grant Google Drive access, then paste your Kaggle API token into the **hidden prompt** (not a notebook cell). Generate the token at https://www.kaggle.com/settings/api.
4. Type `YES` to upload all notebooks to **zeigleritaiml**, **private**, without running them.

No credentials are written to Google Drive. Do not share your token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, subprocess, zipfile, json, re, hashlib, shutil
from pathlib import Path
from getpass import getpass
subprocess.run([sys.executable,'-m','pip','-q','install','-U','kaggle'],check=True)
# Optional: set full path if the automatic search doesn't find your ZIP.
ZIP_PATH = ''
if not ZIP_PATH:
    preferred = ['Kaggle_Notebooks_FLAT_25.zip','Kaggle_Notebooks_Standardized_25.zip','kaggle_zeigleritaiml_ready.zip']
    matches=[]
    for root,dirs,files in os.walk('/content/drive/MyDrive'):
        for fn in files:
            if fn in preferred: matches.append(Path(root)/fn)
    matches.sort(key=lambda p:(preferred.index(p.name),str(p)))
    if not matches: raise FileNotFoundError('ZIP not found in MyDrive. Put it there or set ZIP_PATH above.')
    ZIP_PATH=str(matches[0])
print('Using:',ZIP_PATH)
work=Path('/content/kaggle_upload_work')
if work.exists(): shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        if info.filename.lower().endswith('.ipynb') and not info.is_dir():
            data=z.read(info)
            doc=json.loads(data)
            if 'nbformat' not in doc or 'cells' not in doc: raise ValueError('Invalid notebook: '+info.filename)
            (work/Path(info.filename).name).write_bytes(data)
notebooks=sorted(work.glob('*.ipynb'))
if not notebooks: raise RuntimeError('No .ipynb notebooks in ZIP')
print('Ready:',len(notebooks),'notebooks')
for f in notebooks:print(' •',f.name)

In [ ]:
token=getpass('Paste Kaggle API token (hidden input): ').strip()
if not token: raise ValueError('Kaggle API token is required.')
env=os.environ.copy()
env['KAGGLE_API_TOKEN']=token
del token
owner='zeigleritaiml'
if input(f'Type YES to upload {len(notebooks)} PRIVATE notebooks to {owner}: ') != 'YES':
    raise SystemExit('Cancelled. Nothing uploaded.')
results=[]
for i,f in enumerate(notebooks,1):
    base=re.sub('[^a-z0-9]+','-',f.stem.lower()).strip('-')[:54].rstrip('-')
    slug=f'{base}-{hashlib.sha256(f.name.encode()).hexdigest()[:7]}'
    folder=work/('publish_'+slug)
    folder.mkdir(exist_ok=True)
    shutil.copy2(f,folder/f.name)
    metadata={'id':f'{owner}/{slug}','title':f.stem.replace('_',' ').replace('-',' ')[:100],
              'code_file':f.name,'language':'python','kernel_type':'notebook','is_private':True,
              'enable_gpu':False,'enable_internet':False,'dataset_sources':[],
              'competition_sources':[],'kernel_sources':[],'model_sources':[]}
    (folder/'kernel-metadata.json').write_text(json.dumps(metadata,indent=2))
    print(f'[{i}/{len(notebooks)}] {f.name}',flush=True)
    p=subprocess.run(['kaggle','kernels','push','-p',str(folder),'--no-run'],env=env,capture_output=True,text=True)
    status='SUCCESS' if p.returncode==0 else 'FAILED'
    results.append((status,f.name,f'https://www.kaggle.com/code/{owner}/{slug}'))
    print(status,(p.stdout if p.returncode==0 else p.stderr)[-600:])
print('\nRESULTS:')
for x in results:print(*x)
print('Successful:',sum(x[0]=='SUCCESS' for x in results),'/',len(results))
env.pop('KAGGLE_API_TOKEN',None)